# M13.8 — Descomposición del error de χ_eff entre semillas (val G3)

Pre-registro: `docs/experiments/M13_8_seed_decomposition.md` (4-11-2026, antes de calcular). Solo val G3; sin GPU.

Entradas (solo lectura), en `<data_root>/results/M13_3/<tag>/val_predictions_embeddings.npz` de 6a-aug s123/s124,
4a-aug s123/s124, 4b-aug s123 y 4c-aug s123 (tags de `MODEL_SPECS` de `M13_6_evaluation`). De cada npz solo se leen
`pred_val, y_val, idx_val` y los metadatos pequeños de comprobación (`y_mean, y_std, label_names, split_path`); `emb_val`
no se lee nunca. Del HDF5 G3 solo `target_network_snr` en las filas de val. Escalador train-only del contrato G1
congelado (`load_contract`). Referencia del GATE: `M13_6_evaluation_summary_20261102_1552.json`.

Convenciones como en `M13_6_evaluation`: etiquetas [chirp_mass, total_mass, chi_eff]; unidades físicas; r = f − θ;
deciles de τ30 newtoniano de la M_c verdadera con bordes de cuantiles de val (D1 = τ más corto). Los IC son bootstrap
emparejado por filas de val (B = 2000, `default_rng(13808)`, un único generador consumido en el orden de las celdas):
miden **muestreo de filas, no variación entre semillas**. Ejecutar de arriba abajo una sola vez (regla de parada).
El resumen se escribe en `<data_root>/results/M13_3/M13_8/M13_8_summary_<UTC>.json` (modo `'x'`).

In [ ]:
# 1. Procedencia
import json
import sys
import warnings
from datetime import datetime, timezone
from pathlib import Path

import h5py
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
while not (ROOT / 'AGENTS.md').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.paths import resolve_data_root, resolve_processed_artifact
# Read-only reuse, as in M13_6_evaluation: pinned G1 contract and tau30 of M13.1.
from scripts.m12.m12_5_3_posthoc_check import LABELS, digest, load_contract, npz_to_physical, require
from scripts.m13.m13_1_diagnostics import tau_newtonian

# (key, results tag): same tags as MODEL_SPECS of M13_6_evaluation (cell 1). All six are required here.
MODEL_SPECS = [
    ('6a_aug_s123', 'M13_6a_G3aug_s123'),
    ('6a_aug_s124', 'M13_6a_G3aug_s124'),
    ('4a_aug_s123', 'M13_4a_G3aug_s123'),
    ('4a_aug_s124', 'M13_4a_G3aug_s124'),
    ('4b_aug_s123', 'M13_4b_G3aug_s123'),
    ('4c_aug_s123', 'M13_4c_G3aug_s123'),
]
MODELS = [k for k, _ in MODEL_SPECS]
TAG = dict(MODEL_SPECS)
ARCH_OF = {m: m.rsplit('_s', 1)[0] for m in MODELS}
PAIR_4A, PAIR_6A, M_4B = ('4a_aug_s123', '4a_aug_s124'), ('6a_aug_s123', '6a_aug_s124'), '4b_aug_s123'
WITHIN_PAIRS = [PAIR_4A, PAIR_6A]

cfg = json.loads((ROOT / 'configs/experiments/train_m11_final_G1_500k.json').read_text())
DATA_ROOT = resolve_data_root(config_data_root=cfg.get('data_root'))
M13_DIR = DATA_ROOT / 'results' / 'M13_3'
VAL_NPZ = 'val_predictions_embeddings.npz'
REF_SUMMARY = M13_DIR / 'M13_6_evaluation_summary_20261102_1552.json'
OUT_DIR = M13_DIR / 'M13_8'

# Pre-registered settings (doc M13.8).
N_BOOT = 2000
SEED = 13808
G_THRESHOLD = 0.05                 # primary: G_inf CI vs 5 %
Y_ATOL = 1e-6                      # GATE: max |dy| between models, physical units
REF_RTOL = 1e-6                    # GATE: per-decile RMSE vs M13_6 summary
CHUNK = 100                        # bootstrap replicates per vectorised block (memory only; does not change draws)


def config_of(tag):
    return json.loads((ROOT / 'configs/experiments/m13' / f'train_m{tag[1:]}.json').read_text())


files = {m: M13_DIR / TAG[m] / VAL_NPZ for m in MODELS}
missing = [f'{m}: {p}' for m, p in files.items() if not p.exists()]
require(not missing, f'Missing val npz (required): {missing}')

# Config checks as in M13_6_evaluation cell 3 (shared G3 dataset block, tags, augmentation on).
cfgs = {m: config_of(TAG[m]) for m in MODELS}
ds0 = cfgs[MODELS[0]]['dataset']
require(all(c['dataset'] == ds0 for c in cfgs.values()), 'model configs do not share the G3 dataset block')
for m, c in cfgs.items():
    require(c['outputs']['checkpoint_tag'] == TAG[m], f'{m}: config checkpoint_tag != {TAG[m]}')
    require(bool((c['training'].get('noise_augmentation') or {}).get('enabled', False)), f'{m}: augmentation off')
G3_H5 = resolve_processed_artifact(data_root=DATA_ROOT, dataset_id=ds0['dataset_id'], file_name=ds0['dataset_file'])
require(G3_H5.exists(), f'Missing G3 HDF5: {G3_H5}')

sha = {m: digest(p) for m, p in files.items()}
prov = pd.DataFrame([dict(model=m, tag=TAG[m], path=str(p), size_mb=p.stat().st_size / 2**20,
                          mtime_utc=datetime.fromtimestamp(p.stat().st_mtime, timezone.utc).isoformat(timespec='seconds'),
                          sha256_short=sha[m][:12]) for m, p in files.items()])
print(DATA_ROOT)
print('G3 HDF5:', G3_H5)
print('reference summary:', REF_SUMMARY, '(exists)' if REF_SUMMARY.exists() else '(NOT FOUND)')
prov

In [ ]:
# 2. Carga (solo pred_val, y_val, idx_val + metadatos de comprobación; emb_val no se lee)
paths, splits, c_mean, c_std, identity = load_contract(DATA_ROOT)
checks = ['frozen G1 contract (g1_npe_contract.load_contract)', 'model configs: G3 block, tags, augmentation on']


def check(ok, name):
    require(bool(ok), 'Check failed: ' + name)
    checks.append(name)


def load_pred(path, name):
    # load_pred of M13_6_evaluation (cell 3) without emb_val. NpzFile reads a member only when indexed, so emb_val and
    # the object arrays (model_config, input_normalization) are never decompressed.
    with np.load(path, allow_pickle=False) as a:
        f, y = (a[k].astype(np.float64) for k in ('pred_val', 'y_val'))
        idx = a['idx_val'].astype(np.int64)
        y_mean, y_std = a['y_mean'].astype(np.float64), a['y_std'].astype(np.float64)
        label_names = a['label_names'].tolist()
        split_name = Path(str(a['split_path'])).name
    n = len(idx)
    check(f.shape == y.shape == (n, 3), f'{name}: shapes')
    check(label_names == LABELS, f'{name}: label order {LABELS}')
    check(split_name == paths['split_file'].name, f'{name}: split_path is the pinned split file')
    check(np.allclose(y_mean, c_mean, rtol=1e-6) and np.allclose(y_std, c_std, rtol=1e-6, atol=1e-8),
          f'{name}: npz y_mean/y_std equal M11 train-only label stats')
    check(len(np.unique(idx)) == n, f'{name}: idx without duplicates')
    check(np.isfinite(f).all() and np.isfinite(y).all(), f'{name}: finite values')
    y, f, units = npz_to_physical(y, f, c_mean, c_std)
    check(np.all(np.abs(f.mean(0) - y.mean(0)) < 0.1 * c_std), f'{name}: pred in units of y ({units})')
    o = np.argsort(idx)                # align every model on the HDF5 row id
    return dict(f=f[o], y=y[o], idx=idx[o], units=units)


val = {m: load_pred(files[m], m) for m in MODELS}
pd.DataFrame([dict(model=m, n=len(v['idx']), units=v['units']) for m, v in val.items()])

In [ ]:
# 3. GATE
REF = MODELS[0]
IDX, Y = val[REF]['idx'], val[REF]['y']
check(np.array_equal(IDX, np.sort(splits['val'])), f'{REF}: idx_val equals val membership')
for m in MODELS:
    check(np.array_equal(val[m]['idx'], IDX), f'{m}: same row_ids as {REF} (after sorting by idx_val)')
    dy = np.abs(val[m]['y'] - Y).max(0)
    print(f'{m}: max |y - y_ref| =', dy)
    check(np.all(dy <= Y_ATOL), f'{m}: same y_true as {REF} (max |dy| <= {Y_ATOL:g})')
P = {m: val[m]['f'] for m in MODELS}
R = {m: P[m] - Y for m in MODELS}        # r = f - theta, physical units, columns in LABELS order
N = len(Y)

# tau30 deciles: identical code to M13_6_evaluation cell 9 (Newtonian tau30 of the TRUE Mc, val quantile edges).
tau30 = tau_newtonian(Y[:, 0])
tau_edges = np.quantile(tau30, np.linspace(0, 1, 11))
tdec = np.clip(np.searchsorted(tau_edges[1:-1], tau30, side='right'), 0, 9)
DEC = [f'D{d + 1}' for d in range(10)]
GROUPS = {**{k: tdec == d for d, k in enumerate(DEC)}, 'D1-D3': tdec <= 2, 'D8-D10': tdec >= 7,
          'global': np.ones(N, bool)}

# Per-decile RMSE of every model and label, recomputed.
rmse_dec = pd.DataFrame([dict(model=m, decile=k, n=int(GROUPS[k].sum()),
                              **dict(zip(LABELS, np.sqrt(np.mean(R[m][GROUPS[k]]**2, 0)))))
                         for k in DEC for m in MODELS])
RM = rmse_dec.set_index(['decile', 'model'])


def rel_ok(a, b):
    return abs(a - b) <= REF_RTOL * abs(b)


ref = json.loads(REF_SUMMARY.read_text()) if REF_SUMMARY.exists() else {}
fails, n_cmp = [], 0
if 'tau_edges_s' in ref:
    check(np.allclose(tau_edges, ref['tau_edges_s'], rtol=REF_RTOL, atol=0), 'tau30 decile edges equal M13_6 summary')
# seed_noise_by_decile.per_model (M13_6 cell 11): per-model chi_eff RMSE and n per tau30 decile.
for row in ref.get('seed_noise_by_decile', {}).get('per_model', []):
    if row.get('grouping') == 'tau30' and row['model'] in MODELS:
        n_cmp += 1
        t = RM.loc[(row['cls'], row['model'])]
        if not (int(t['n']) == row['n'] and rel_ok(t['chi_eff'], row['rmse'])):
            fails.append(('per_model', row['model'], row['cls'], t['chi_eff'], row['rmse']))
# tau_deciles (M13_6 cell 9): rmse_model = mean RMSE over the seeds in 'seeds'; rmse_6a = mean over settings.a6.
a6_ref = ref.get('settings', {}).get('a6') or []
for row in ref.get('tau_deciles', []):
    k, lab, seeds = f"D{row['tau_decile']}", row['label'], row['seeds'].split('+')
    for name, ms, v in (('rmse_model', seeds, row['rmse_model']), ('rmse_6a', a6_ref, row['rmse_6a'])):
        if ms and all(s in MODELS for s in ms):
            n_cmp += 1
            x = np.mean([RM.loc[(k, s), lab] for s in ms])
            if not rel_ok(x, v):
                fails.append((name, '+'.join(ms), k, lab, x, v))
with pd.option_context('display.float_format', '{:.6g}'.format, 'display.width', 200):
    display(rmse_dec.pivot(index='decile', columns='model', values='chi_eff').loc[DEC])
if n_cmp == 0:
    warnings.warn(f'{REF_SUMMARY.name}: no per-decile RMSE to compare; table printed above, check it by hand')
else:
    require(not fails, f'Per-decile RMSE differs from {REF_SUMMARY.name} (rtol {REF_RTOL:g}): {fails[:10]}')
    checks.append(f'per-decile RMSE equals {REF_SUMMARY.name} ({n_cmp} values, rtol {REF_RTOL:g})')

# SNR as in M13_6_evaluation (cell 3 / section 5): target_network_snr of the G3 HDF5 at the sorted val rows.
with h5py.File(G3_H5, 'r') as h:
    require('target_network_snr' in h, 'G3 HDF5 lacks target_network_snr')
    SNR = h['target_network_snr'][IDX].astype(np.float64)
check(np.isfinite(SNR).all() and np.all(SNR > 0), 'target_network_snr finite and positive')
print('n_val:', N, '| decile sizes:', [int(GROUPS[k].sum()) for k in DEC])
print('GATE passed:', len(checks), 'checks')

In [ ]:
# 4. Funciones y bootstrap
QUANTITIES = ['mse_a', 'mse_b', 'mse_single', 'Ed2', 'dbar', 'Ed2_centr', 'phi', 'G2', 'Ginf', 'common', 'rho_unc',
              'pearson']


def pair_stats(ra, rb):
    # Pre-registered quantities for residuals ra, rb (r = f - theta) on the same rows. Reduces over the last axis, so
    # (replicates, rows) inputs give one value per replicate. Note 1 - phi = common / mse_single: G_inf is nan if common < 0.
    mse_a, mse_b = np.mean(ra**2, -1), np.mean(rb**2, -1)
    mse_single = 0.5 * (mse_a + mse_b)
    d, m = 0.5 * (ra - rb), 0.5 * (ra + rb)
    Ed2 = np.mean(d**2, -1)
    phi = 2.0 * Ed2 / mse_single
    common = np.mean(ra * rb, -1)
    ca, cb = ra - ra.mean(-1, keepdims=True), rb - rb.mean(-1, keepdims=True)
    with np.errstate(invalid='ignore'):
        g_inf = 1.0 - np.sqrt(1.0 - phi)
    return dict(mse_a=mse_a, mse_b=mse_b, mse_single=mse_single, Ed2=Ed2, dbar=np.mean(d, -1), Ed2_centr=np.var(d, -1),
                phi=phi, G2=1.0 - np.sqrt(np.mean(m**2, -1) / mse_single), Ginf=g_inf, common=common,
                rho_unc=common / np.sqrt(mse_a * mse_b),
                pearson=np.mean(ca * cb, -1) / np.sqrt(np.mean(ca**2, -1) * np.mean(cb**2, -1)))


rng = np.random.default_rng(SEED)        # single generator, consumed in cell order (cells 5-11)


def boot(fn, arrays, rows):
    # Point estimate and paired row bootstrap of fn(*arrays) on boolean mask `rows`: one index draw per replicate,
    # shared by every array and every quantity returned by fn. Blocks of CHUNK replicates only bound memory.
    xs = [a[rows] for a in arrays]
    n = len(xs[0])
    point = {k: float(v) for k, v in fn(*xs).items()}
    reps = {k: [] for k in point}
    for s in range(0, N_BOOT, CHUNK):
        ii = rng.integers(0, n, size=(min(CHUNK, N_BOOT - s), n))
        out = fn(*(x[ii] for x in xs))
        for k in reps:
            reps[k].append(out[k])
    return point, {k: np.concatenate(v) for k, v in reps.items()}


def ci(b):
    return np.percentile(b, [2.5, 97.5])  # nan if any replicate is nan (reported as n_nan)


def rows_of(point, reps, **keys):
    return [dict(**keys, quantity=q, value=point[q], ci95_lo=ci(reps[q])[0], ci95_hi=ci(reps[q])[1],
                 n_nan=int(np.isnan(reps[q]).sum())) for q in point]


def wide(t, index='group'):
    w = t.pivot(index=index, columns='quantity', values=['value', 'ci95_lo', 'ci95_hi'])
    w.columns = [q if s == 'value' else f'{q}_{s[5:]}' for s, q in w.columns]
    order = [c for q in t.quantity.unique() for c in (q, f'{q}_lo', f'{q}_hi')]
    return w.loc[t[index].unique(), order]


PAIR_GROUPS = DEC + ['D1-D3', 'D8-D10']


def pair_analysis(pair, label):
    j, (a, b) = LABELS.index(label), pair
    rows, reps = [], {}
    for k in PAIR_GROUPS:
        g = GROUPS[k]
        pt, reps[k] = boot(pair_stats, [R[a][:, j], R[b][:, j]], g)
        rows += rows_of(pt, reps[k], pair=f'{a}/{b}', label=label, group=k, n=int(g.sum()))
    return pd.DataFrame(rows), reps


def s0(tab, reps):
    # D10 and D1-D3 are disjoint row sets, each with its own independent draws from `rng` (pair_analysis):
    # replicate k of one group is independent of replicate k of the other.
    v = tab.set_index(['group', 'quantity']).value
    b = reps['D10']['phi'] - reps['D1-D3']['phi']
    lo, hi = ci(b)
    return dict(pair=tab.pair.iloc[0], label=tab.label.iloc[0], phi_D10=v[('D10', 'phi')],
                phi_D1_D3=v[('D1-D3', 'phi')], diff=v[('D10', 'phi')] - v[('D1-D3', 'phi')], ci95_lo=lo, ci95_hi=hi,
                excludes_0=bool(lo > 0 or hi < 0))


def s1_stats(r4b, ra, rb):
    mse_4b, common = np.mean(r4b**2, -1), np.mean(ra * rb, -1)
    return dict(mse_4b=mse_4b, common_4a=common, delta=mse_4b - common, ratio=mse_4b / common)


def s1(label, groups=('D10', 'D1-D3')):
    j, rows = LABELS.index(label), []
    for k in groups:
        g = GROUPS[k]
        pt, bt = boot(s1_stats, [R[M_4B][:, j], R[PAIR_4A[0]][:, j], R[PAIR_4A[1]][:, j]], g)
        rows += rows_of(pt, bt, label=label, group=k, n=int(g.sum()))
    return pd.DataFrame(rows)

In [ ]:
# 5. Primario: par 4a-aug s123/s124, chi_eff, D1..D10, D1-D3, D8-D10
tab_primary, reps_primary = pair_analysis(PAIR_4A, 'chi_eff')
with pd.option_context('display.float_format', '{:.4g}'.format, 'display.width', 250, 'display.max_columns', 60):
    display(wide(tab_primary))
print('bootstrap replicates with nan:', tab_primary.groupby('quantity').n_nan.sum().to_dict())

In [ ]:
# 6. S0: phi(D10) - phi(D1-D3), par 4a-aug
s0_4a = s0(tab_primary, reps_primary)
pd.Series(s0_4a)

In [ ]:
# 7. S1: mse_4b - common_4a y mse_4b / common_4a, chi_eff, D10 y D1-D3 (bootstrap emparejado de filas)
tab_s1 = s1('chi_eff')
with pd.option_context('display.float_format', '{:.5g}'.format, 'display.width', 200):
    display(wide(tab_s1))

In [ ]:
# 8. S2: matriz 6x6 de rho_unc (chi_eff) en D1, D5, D10 y global; dentro frente a entre arquitecturas
J = LABELS.index('chi_eff')
S2_GROUPS = ['D1', 'D5', 'D10', 'global']
rho_mats, rows_s2 = {}, []
for k in S2_GROUPS:
    g = GROUPS[k]
    M = pd.DataFrame(np.eye(len(MODELS)), index=MODELS, columns=MODELS)
    for i, a in enumerate(MODELS):
        for b in MODELS[i + 1:]:
            M.loc[a, b] = M.loc[b, a] = pair_stats(R[a][g, J], R[b][g, J])['rho_unc']
    rho_mats[k] = M
    between = [M.loc[a, b] for i, a in enumerate(MODELS) for b in MODELS[i + 1:] if ARCH_OF[a] != ARCH_OF[b]]
    rows_s2.append(dict(group=k, n=int(g.sum()), rho_within_4a=M.loc[PAIR_4A], rho_within_6a=M.loc[PAIR_6A],
                        rho_within_mean=np.mean([M.loc[p] for p in WITHIN_PAIRS]), rho_between_mean=np.mean(between),
                        rho_between_min=np.min(between), rho_between_max=np.max(between), n_between=len(between)))
tab_s2 = pd.DataFrame(rows_s2)
with pd.option_context('display.float_format', '{:.4f}'.format, 'display.width', 200):
    for k, M in rho_mats.items():
        print(k)
        display(M)
    display(tab_s2)

In [ ]:
# 9. S3: terciles de SNR dentro de D10 y de D1-D3 (cortes de val dentro de cada grupo), par 4a-aug, chi_eff
rows_s3, s3_slope = [], {}
a, b = PAIR_4A
for k in ('D10', 'D1-D3'):
    g = GROUPS[k]
    cuts = np.quantile(SNR[g], [1 / 3, 2 / 3])
    ter = np.searchsorted(cuts, SNR, side='right')       # 0, 1, 2; only rows in g are used
    for t in range(3):
        rt = g & (ter == t)
        st = pair_stats(R[a][rt, J], R[b][rt, J])
        rows_s3.append(dict(group=k, tercile=t + 1, n=int(rt.sum()), snr_lo=SNR[rt].min(), snr_hi=SNR[rt].max(),
                            snr_median=np.median(SNR[rt]), common_4a=st['common'], mse_single=st['mse_single']))
    t = [r for r in rows_s3 if r['group'] == k]
    c = np.array([r['common_4a'] for r in t])
    # Descriptive 3-point least-squares fit; undefined if any common <= 0.
    s3_slope[k] = (float(np.polyfit(np.log([r['snr_median'] for r in t]), np.log(c), 1)[0])
                   if np.all(c > 0) else float('nan'))
tab_s3 = pd.DataFrame(rows_s3)
with pd.option_context('display.float_format', '{:.5g}'.format, 'display.width', 200):
    display(tab_s3)
print('slope d log(common_4a) / d log(SNR median):', s3_slope)

In [ ]:
# 10. S4: celdas 5-6 para el par 6a-aug s123/s124 (chi_eff)
tab_s4, reps_s4 = pair_analysis(PAIR_6A, 'chi_eff')
s0_6a = s0(tab_s4, reps_s4)
with pd.option_context('display.float_format', '{:.4g}'.format, 'display.width', 250, 'display.max_columns', 60):
    display(wide(tab_s4))
pd.Series(s0_6a)

In [ ]:
# 11. S5: celdas 5 y 7 para chirp_mass y total_mass
tab_s5, tab_s5_s1 = {}, {}
for lab in ('chirp_mass', 'total_mass'):
    tab_s5[lab], _ = pair_analysis(PAIR_4A, lab)
    tab_s5_s1[lab] = s1(lab)
with pd.option_context('display.float_format', '{:.4g}'.format, 'display.width', 250, 'display.max_columns', 60):
    for lab in tab_s5:
        print(lab)
        display(wide(tab_s5[lab]))
        display(wide(tab_s5_s1[lab]))

In [ ]:
# 12. Veredicto del primario (reglas del pre-registro)
p = tab_primary.set_index(['group', 'quantity']).loc[('D10', 'Ginf')]
g_lo, g_hi = float(p.ci95_lo), float(p.ci95_hi)
if g_lo >= G_THRESHOLD:
    verdict = 'varianza relevante'
elif g_hi < G_THRESHOLD:
    verdict = 'varianza secundaria'
else:
    verdict = 'no concluyente'      # includes a nan CI (some replicate with common < 0)
s1_d10 = tab_s1.set_index(['group', 'quantity']).loc[('D10', 'delta')]
readings = dict(
    primary=dict(verdict=verdict, pair='/'.join(PAIR_4A), label='chi_eff', group='D10', Ginf=float(p.value),
                 ci95=[g_lo, g_hi], n_nan=int(p.n_nan), threshold=G_THRESHOLD),
    S0_4a=dict(**s0_4a, reading='inestabilidad específica de señales largas' if s0_4a['excludes_0'] else 'IC incluye 0'),
    S1_D10=dict(delta=float(s1_d10.value), ci95=[float(s1_d10.ci95_lo), float(s1_d10.ci95_hi)],
                reading=('ventaja de 4b no alcanzable promediando 4a' if s1_d10.ci95_hi < 0 else 'IC no queda por debajo de 0')),
    S2=tab_s2[['group', 'rho_within_4a', 'rho_within_mean', 'rho_between_mean']].to_dict('records'),
    S3_slope=s3_slope,
    S0_6a=s0_6a,
)
print(f"PRIMARIO (chi_eff, D10, 4a-aug s123/s124): G_inf = {p.value:.4f} [{g_lo:.4f}, {g_hi:.4f}] -> {verdict.upper()}")
readings

In [ ]:
# 13. Resumen JSON (no sobrescribe: modo 'x')
def to_json(o):
    if isinstance(o, np.integer):
        return int(o)
    if isinstance(o, np.floating):
        return float(o)
    if isinstance(o, np.bool_):
        return bool(o)
    if isinstance(o, np.ndarray):
        return o.tolist()
    return str(o)


stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
OUT_DIR.mkdir(parents=True, exist_ok=True)
SUMMARY_PATH = OUT_DIR / f'M13_8_summary_{stamp}.json'
summary = dict(
    experiment='M13.8 seed decomposition of the error (val G3 only)',
    preregistration='docs/experiments/M13_8_seed_decomposition.md (4-11-2026)',
    created_utc=stamp,
    models={m: dict(tag=TAG[m], path=str(files[m]), sha256=sha[m]) for m in MODELS},
    reference_summary=dict(path=str(REF_SUMMARY), exists=REF_SUMMARY.exists(), n_compared=n_cmp),
    g3_hdf5=dict(path=str(G3_H5), columns=['target_network_snr'], rows='val only'),
    contract_identity=identity,
    checks=checks,
    settings=dict(labels=LABELS, residual='f - theta', units='physical (train-only M11 scaler)', n_val=N,
                  n_boot=N_BOOT, seed=SEED, bootstrap='paired row resampling of val: row sampling, not seed variation',
                  rng='single default_rng(13808) consumed in cell order 5-11', y_atol=Y_ATOL, ref_rtol=REF_RTOL,
                  tau_deciles='tau30 Newtonian from true Mc, val quantile edges (M13_6 cell 9)', tau_edges_s=tau_edges,
                  snr='target_network_snr (G3 HDF5); S3 tercile cuts from val rows within each group',
                  pair_4a=PAIR_4A, pair_6a=PAIR_6A, model_4b=M_4B, g_threshold=G_THRESHOLD),
    rmse_by_decile=rmse_dec.to_dict('records'),
    primary=tab_primary.to_dict('records'),
    s0_4a=s0_4a,
    s1_chi_eff=tab_s1.to_dict('records'),
    s2_rho_unc={k: M.to_dict() for k, M in rho_mats.items()},
    s2_summary=tab_s2.to_dict('records'),
    s3=tab_s3.to_dict('records'),
    s3_slope=s3_slope,
    s4_6a=tab_s4.to_dict('records'),
    s4_s0_6a=s0_6a,
    s5={lab: dict(pair=tab_s5[lab].to_dict('records'), s1=tab_s5_s1[lab].to_dict('records')) for lab in tab_s5},
    readings=readings,
)
with open(SUMMARY_PATH, 'x') as fh:
    json.dump(summary, fh, indent=1, default=to_json)
print('saved', SUMMARY_PATH)